In [0]:
%run ./00_config

In [0]:
%run ./02_core_functions

# 03_main_nphc - monthly
For every active **NPHC** row in the control table:

1. **Copy** zip files from the source to `<zip_tgt_path>/<mid_date>/`
2. **Unzip** them into `<unzip_tgt_path>/<mid_date>/<zip_name>/`
3. **Ingest** the JSON files into the target table, tagging each row with its `zip_file`

**First run:** every zip at the source is loaded.
**Later runs:** only *new* zips (not yet in `raw/`). A month that was missed is simply still "new", so it is loaded on the next run together with the current month. Set `is_full_load = 1` in the control table to force everything again.

If a zip fails to unzip or ingest, its raw copy is removed so the next run retries it. Re-ingesting a file replaces its previous row, so there are never duplicates.

In [0]:
run_id          = str(uuid.uuid4())
control_entries = get_active_control_entries("NPHC")
ctl_by_sno      = {c["s_no"]: c for c in control_entries}
log_records     = []

print(f"run_id = {run_id}")
print(f"{len(control_entries)} active NPHC control row(s)")

### Preview - which months will be loaded?

In [0]:
this_month = datetime.now().strftime("%Y%m")
for ctl in control_entries:
    all_zips = list_zip_files(ctl["zip_source_path"])
    zips     = select_zips_to_copy(all_zips, ctl)
    print(f"s_no={ctl['s_no']}: {len(zips)} new zip(s); months (end date) to load: {nphc_months(zips) or '-'}")
    if this_month not in nphc_months(all_zips):
        print(f"  WARNING: no zip for the current month {this_month} at the source yet")

### Stage 1 + 2 - Copy and Unzip

In [0]:
copy_and_unzip(run_id, control_entries, log_records)

### Stage 3 - Ingest JSON

In [0]:
ingest_summary = {"SUCCESS": 0, "FAILED": 0}

unzipped = [r for r in log_records if r["stage"] == "UNZIP" and r["status"] == "SUCCESS"]
print(f"{len(unzipped)} unzipped zip file(s) to ingest")

for unzip_rec in unzipped:
    print(f"--- {unzip_rec['file_name']} ---")
    counts = nphc_ingest_zip(unzip_rec, ctl_by_sno[unzip_rec["s_no"]], run_id)
    for k, v in counts.items():
        ingest_summary[k] += v
    if counts["FAILED"] > 0:
        discard_raw_copy(unzip_rec["source_path"])      # retried on the next run

print("\nJSON ingestion summary:", ingest_summary)

### Run summary

In [0]:
display(spark.table(log_table).filter(F.col("run_id") == run_id).orderBy("start_time"))

In [0]:
display(spark.table(load_log_table).filter(F.col("run_id") == run_id).orderBy("started_at"))

In [0]:
# Result: one row per JSON file, with the zip it came from
for t in {f"{c['tgt_catalog']}.{c['tgt_schema']}.{c['tgt_table']}" for c in control_entries}:
    if spark.catalog.tableExists(t):
        display(spark.table(t).select("zip_file", "file_id", "file_name", "dt_datafile", "file_size_mb", "table_name")
                              .orderBy("file_id"))

In [0]:
# Fail the notebook / job if anything failed (details are in the log tables for this run_id)
failed = sum(1 for r in log_records if r["status"] == "FAILED") + ingest_summary["FAILED"]
if failed > 0:
    raise Exception(f"{failed} step(s) failed. See {log_table} / {load_log_table} where run_id = '{run_id}'")

In [0]:
%sql
SELECT * FROM iics.casemix.nphc_json_test